In [ ]:
pip install --upgrade scikit-learn

In [ ]:
pip install numpy==1.26.4 scikit-learn pandas

In [ ]:
pip install numpy==1.26.4 --force-reinstall

In [ ]:
print(f"NumPy version: {np.__version__}")

In [ ]:
import sklearn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

In [ ]:
import json
import ast
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
directory_data = Path("/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset")
business_json = directory_data /"yelp_academic_dataset_business.json"
review_json = directory_data /"yelp_academic_dataset_review.json"
checkin_json = directory_data /"yelp_academic_dataset_checkin.json"

In [ ]:
df = pd.read_json(business_json, lines=True)
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

counts = df['is_open'].value_counts().sort_index()

plt.figure(figsize=(6,4))
sns.barplot(x=counts.index, y=counts.values, palette=['#d9534f', '#5cb85c'])
plt.xlabel('is_open (0 = закрыто, 1 = открыто)')
plt.ylabel('кол-во заведений')
plt.xticks([0,1])
plt.show()

In [ ]:
restaurants_df = df[df['categories'].str.contains('Restaurants', case=False, na=False)]
r_ids = set(restaurants_df['business_id'])

chunks = []
for ch in pd.read_json(review_json, lines=True, chunksize=100000):
    ch = ch[ch['business_id'].isin(r_ids)].copy()
    if not ch.empty:
        ch['date'] = pd.to_datetime(ch['date'])
        chunks.append(ch)

rev1 = pd.concat(chunks, ignore_index=True)
rev1.shape

In [ ]:
rev1 = rev1[['business_id', 'text', 'stars', 'date']]
df = df[['business_id', 'is_open']]

In [ ]:
df1 = rev1.merge(df, on='business_id', how='inner')
df1

In [ ]:
df1['date'] = pd.to_datetime(df1['date'])

last_review_date = df1.groupby('business_id')['date'].max().reset_index()
last_review_date.rename(columns={'date': 'last_review_date'}, inplace=True)

df1 = df1.merge(last_review_date, on='business_id', how='left')

In [ ]:
df1['cutoff_date'] = df1['last_review_date'] - pd.Timedelta(days=365)

df_filtered = df1[df1['date'] >= df1['cutoff_date']].copy()

In [ ]:
df_filtered.drop(columns=['cutoff_date', 'last_review_date'], inplace=True)

In [ ]:
df_filtered.shape

In [ ]:
df_filtered.to_csv('filtered_rew.csv', index=False, encoding='utf-8')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
rev1['text_len'] = rev1['text'].str.len() 
corr = rev1[['useful','funny','cool','text_len']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Корреляция между голосами и длиной текста')
plt.show()

In [ ]:
rev1['date'] = pd.to_datetime(rev1['date'])
rev1['year'] = rev1['date'].dt.year
rpy = rev1.groupby('year').size()
plt.figure(figsize=(12,6))
sns.lineplot(x=rpy.index, y=rpy.values, marker='o')
plt.title('Динамика количества отзывов по годам')
plt.xlabel('Год')
plt.ylabel('Количество отзывов')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
rev1['rating_group'] = '4-5 (положительные)'
rev1.loc[rev1['stars'] <= 3.5, 'rating_group'] = '1-3 (отрицательные)'

group_counts = rev1['rating_group'].value_counts().reindex(['1-3 (отрицательные)', '4-5 (положительные)'])

plt.figure(figsize=(8,6))
sns.barplot(x=group_counts.index, y=group_counts.values, palette=['red', 'green'])
plt.xlabel('Группа оценки')
plt.ylabel('Количество отзывов')
plt.show()

In [ ]:
rev.text.describe()

In [ ]:
rev1.drop_duplicates(subset='text', keep='first', inplace=True)

In [ ]:
rev1.shape

In [ ]:
rev1.to_csv('rev11.csv', index=False, encoding='utf-8')

In [ ]:
!pip install sweetviz

In [ ]:
# еще один инструмент автоматического EDA, я экспериментировала 
import sweetviz as sv
report = sv.analyze(rev1)
report.show_html()

In [ ]:
pip install torch==2.2.2 torchaudio==2.2.2 torchvision==0.17.2 torchtext==0.17.2 torchdata==0.11.0

In [ ]:
df_filtered.text.unique()

In [ ]:
df_filtered=pd.read_csv('filtered_rew.csv', encoding='utf-8')
df_filtered

In [ ]:
import re
df_filtered['text_clean'] = df_filtered['text'].str.lower().str.replace(r'[^a-z0-9\s\!\?]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
df_filtered.text_clean.unique()

In [ ]:
from transformers import BertTokenizerFast

In [ ]:
import torch

In [ ]:
review_counts = df_filtered['business_id'].value_counts()
valid_business_ids = review_counts[review_counts >= 10].index
df_final = df_filtered[df_filtered['business_id'].isin(valid_business_ids)].copy().reset_index()

In [ ]:
df_final

In [ ]:
print(tokenizer.save_pretrained(".")) # сохранила для гита на всякий

In [ ]:
df_final.token_len.max()

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install numpy==1.26.4
!pip install transformers

In [ ]:
from tqdm import tqdm
# чтобы я могла видеть прогресс, а то я че то не вкуривала оно вообще работает или нет

# я применяю быструю BERT токенизацию
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

texts = df_final['text_clean'].astype(str).tolist()

batch_size = 1000 # поделила на батчи, чтобы быстрее работал, а то даже на 600к строках долго было
lengths = [] # для статистики длина токена

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    encodings = tokenizer(batch, add_special_tokens=True, truncation=False, padding=False)
    for ids in encodings['input_ids']:
        lengths.append(len(ids))

df_final['token_len'] = lengths

In [ ]:
df_final

In [ ]:
# тут я уже создаю сами токены и маску для будущего обучения и кладу в датафрейм
ids_list = []
masks_list = []
max_len = int(df_final['token_len'].quantile(0.95)) # тк я не знаю какую длину взять, то возьмем такой перцентиль, чтобы не обрезать сильно слишком длинные отзывы

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    encodings = tokenizer(batch, add_special_tokens=True, truncation=True, max_length=max_len, padding='max_length', return_tensors='pt')
    ids_list.append(encodings['input_ids'])
    masks_list.append(encodings['attention_mask'])

ids = torch.cat(ids_list, dim=0) # dim нужен, чтобы он склеил батчи
masks = torch.cat(masks_list, dim=0)

df_final['input_ids'] = ids.tolist()
df_final['attention_mask'] = masks.tolist()

In [ ]:
# сделаем тут колонку для разметки, типа 1 - хороший отзыв, 0 - плохой по кол-ву звезд 

df_final['label'] = (df_final['stars'] >= 4).astype(int)

# p.c оно не пригодится лол

In [ ]:
df_final

In [ ]:
# посмортрим наши метрики, может что интересное будет
metrics=df_final['token_len']

stats = {
    'count': len(metrics),
    'mean': metrics.mean(),
    'std': metrics.std(),
    'min': metrics.min(),
    '25%': metrics.quantile(0.25),
    '50%': metrics.quantile(0.50),
    '75%': metrics.quantile(0.75),
    '90%': metrics.quantile(0.90),
    '95%': metrics.quantile(0.95),
    '99%': metrics.quantile(0.99),
    'max': metrics.max()
}

metrics_df = pd.DataFrame([stats])
metrics_df

In [ ]:
import torch.nn as nn

vocab_size = tokenizer.vocab_size
embed_dim = 100 # берем 100, чтобы не расходовать сильно память, но получить норм результат
padding_idx = 0

embedding_layer = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)

In [ ]:
# выделим теперь биграммы и триграммы, чтобы потом классифицировать, какие относятся к негативным отзывам, а какие к положительным
# они нам понадобятся для выявления проблемных фраз
import sklearn
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(2,3), max_features=100, stop_words='english')
X_ngrams = vectorizer.fit_transform(df_final['text_clean'])
feature_names = vectorizer.get_feature_names_out()

In [ ]:
counts = np.asarray(X_ngrams.sum(axis=0)).flatten()
idx = np.argsort(counts)[-70:][::-1]
ngrams = feature_names[idx]
freq = counts[idx]

plt.figure(figsize=(10, 12))
plt.barh(ngrams[::-1], freq[::-1], color='skyblue')
plt.xlabel('Частота')
plt.title('Топ-70')
plt.tight_layout()
plt.show()

In [ ]:
pos_mask = (df_final['label'] == 1).values
neg_mask = (df_final['label'] == 0).values
X_arr = X_ngrams.toarray()

pos_counts = X_arr[pos_mask].sum(axis=0)[idx]
neg_counts = X_arr[neg_mask].sum(axis=0)[idx]

y_pos = np.arange(len(ngrams))
plt.figure(figsize=(10, 12))
plt.barh(y_pos - 0.2, pos_counts[::1], 0.4, label='Позитивные (label=1)', color='green')
plt.barh(y_pos + 0.2, neg_counts[::1], 0.4, label='Негативные (label=0)', color='red')
plt.yticks(y_pos, ngrams[::-1])
plt.xlabel('Частота')
plt.title('Топ-70')
plt.legend()
plt.show()

In [ ]:
# если честно он как-то не очень нашел биграммы и триграммы
# я не знаю почему((((
# надо как-то почистить их что ли.... но наверное не вручную
from scipy.sparse import csr_matrix

top5_ngrams = []
for i in range(X_ngrams.shape[0]):
    row = X_ngrams[i].toarray().flatten()
    present = [feature_names[j] for j in range(len(feature_names)) if row[j] > 0]
    top5_ngrams.append(present[:5])

df_final['top5_ngrams'] = top5_ngrams

In [ ]:
df_final

In [ ]:
df_final.to_csv('final.csv', encoding='utf-8',  index=False,)

In [ ]:
df_final=pd.read_csv('/kaggle/input/datasets/fishshell/final-file/final (1).csv', encoding='utf-8')

In [ ]:
df_final

In [ ]:
# я не могла скачать норм датасет отсюда, какой-то кринж лол
from IPython.display import FileLink
FileLink('filtered_after_token.csv')

In [ ]:
pip install -q mlflow

In [ ]:
import mlflow
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.pytorch

In [ ]:
from pathlib import Path

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("bilstm")  

use_mlflow = True  
artifact_dir = Path("mlflow_artifacts")
artifact_dir.mkdir(exist_ok=True)

In [ ]:
from torch.utils.data import Dataset
import torch.nn as nn

# это уже классы для обучения, почти все копипаст из доки 

class RestaurantDataset(Dataset):
    def __init__(self, df):
        self.input_ids = torch.tensor(np.array(df['input_ids'].tolist()), dtype=torch.long)
        self.attention_mask = torch.tensor(np.array(df['attention_mask'].tolist()), dtype=torch.long)
        self.is_open = torch.tensor(df['is_open'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.is_open)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'is_open': self.is_open[idx]
        }
        
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        emb = self.embedding(input_ids)
        _, (hidden, _) = self.lstm(emb)
        hidden_last = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(self.dropout(hidden_last))

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    for batch in tqdm(loader, desc="Train"):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['is_open'].to(device).long()
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    return avg_loss, acc, prec, rec, f1

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval"):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['is_open'].to(device).long()
            logits = model(ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return avg_loss, acc, prec, rec, f1, auc

In [ ]:
from sklearn.model_selection import train_test_split
import ast

# потому что оно как строка в таблице, а такое не подходит
df_final['input_ids'] = df_final['input_ids'].apply(ast.literal_eval)
df_final['attention_mask'] = df_final['attention_mask'].apply(ast.literal_eval)

train_df, temp_df = train_test_split(df_final, test_size=0.2, random_state=42, stratify=df_final['is_open'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['is_open'])

all_ids = np.concatenate(df_final['input_ids'].values)
vocab_size_real = int(max(all_ids)) + 1
print(f"Vocab size: {vocab_size_real}")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# гиперпараметры отдельно
import torch

config = {
    "embed_dim": 100,
    "hidden_dim": 128,
    "num_lstm_layers": 2,
    "dropout": 0.5,
    "batch_size": 64,
    "learning_rate": 1e-3,
    "epochs": 5,
    "vocab_size": vocab_size_real,  
    "max_seq_len": 128,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

In [ ]:
from torch.utils.data import DataLoader

train_dataset = RestaurantDataset(train_df)
val_dataset = RestaurantDataset(val_df)
test_dataset = RestaurantDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

In [ ]:
from tqdm import tqdm

run_name = f"BiLSTM_emb{config['embed_dim']}_hid{config['hidden_dim']}_layers{config['num_lstm_layers']}_lr{config['learning_rate']}_bs{config['batch_size']}"

with mlflow.start_run(run_name=run_name) as run:
#тут я логирую параметры из конфига, их можно там менять
    mlflow.log_params({
        "model_type": "BiLSTM",
        "embed_dim": config['embed_dim'],
        "hidden_dim": config['hidden_dim'],
        "num_layers": config['num_lstm_layers'],
        "dropout": config['dropout'],
        "batch_size": config['batch_size'],
        "learning_rate": config['learning_rate'],
        "epochs": config['epochs'],
        "vocab_size": config['vocab_size'],
        "max_seq_len": config['max_seq_len']
    })
    
# фкнции потерь всякие и оптимизация...
    model = BiLSTMClassifier(
        vocab_size=config['vocab_size'],
        embed_dim=config['embed_dim'],
        hidden_dim=config['hidden_dim'],
        num_layers=config['num_lstm_layers'],
        num_classes=2,
        dropout=config['dropout']
    ).to(config['device'])
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    criterion = nn.CrossEntropyLoss()
    
    best_val_f1 = 0.0
    best_model_state = None
    history = {"train_loss": [], "train_f1": [], "val_loss": [], "val_f1": [], "val_auc": []}
    
    for epoch in range(1, config['epochs']+1):
        train_loss, train_acc, train_prec, train_rec, train_f1 = train_epoch(
            model, train_loader, optimizer, criterion, config['device']
        )
        val_loss, val_acc, val_prec, val_rec, val_f1, val_auc = eval_epoch(
            model, val_loader, criterion, config['device']
        )
        
# метрики для разных эпох
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "train_precision": train_prec,
            "train_recall": train_rec,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "val_precision": val_prec,
            "val_recall": val_rec,
            "val_f1": val_f1,
            "val_roc_auc": val_auc
        }, step=epoch)
        
        # а это короче надо, чтобы потом видеть отчеты mlflow с графиками
        history["train_loss"].append(train_loss)
        history["train_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)
        history["val_auc"].append(val_auc)
        
        print(f"Epoch {epoch}: train_loss={train_loss:.4f} train_f1={train_f1:.4f} | val_loss={val_loss:.4f} val_f1={val_f1:.4f} val_auc={val_auc:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = model.state_dict().copy()

    model.load_state_dict(best_model_state)
    
# тест
    test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = eval_epoch(
        model, test_loader, criterion, config['device']
    )
    test_metrics = {
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_precision": test_prec,
        "test_recall": test_rec,
        "test_f1": test_f1,
        "test_roc_auc": test_auc
    }
    mlflow.log_metrics(test_metrics)
    
# вот эти графики, мб удалю потом, они из доки
    os.makedirs("plots", exist_ok=True)
    plt.figure()
    plt.plot(range(1, config['epochs']+1), history["train_loss"], label='Train Loss')
    plt.plot(range(1, config['epochs']+1), history["val_loss"], label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid()
    plt.savefig("plots/loss_curve.png"); 
    
    plt.figure()
    plt.plot(range(1, config['epochs']+1), history["train_f1"], label='Train F1')
    plt.plot(range(1, config['epochs']+1), history["val_f1"], label='Val F1')
    plt.xlabel('Epoch'); plt.ylabel('F1'); plt.legend(); plt.grid()
    plt.savefig("plots/f1_curve.png"); 
    
    plt.figure()
    plt.plot(range(1, config['epochs']+1), history["val_auc"], label='Val ROC-AUC', color='green')
    plt.xlabel('Epoch'); plt.ylabel('AUC'); plt.legend(); plt.grid()
    plt.savefig("plots/auc_curve.png"); 
    
    mlflow.log_artifacts("plots", artifact_path="metric_plots")
    
# логирование из доки
    mlflow.pytorch.log_model(model, "bilstm_model")
    
    print(f"Run ID: {run.info.run_id}")
    print(f"Лучший val F1: {best_val_f1:.4f}, тестовый F1: {test_f1:.4f}")

In [ ]:
mlflow ui --backend-store-uri sqlite:///mlflow.db